# DCN-v2 Training (Minimal GPU Notebook)

All logic is in modules. This notebook only runs GPU training.

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set project path
PROJECT_PATH = '/content/drive/MyDrive/learning/MBA/Course/Deep Learning/final_ex'

import sys, os
sys.path.insert(0, f'{PROJECT_PATH}/src')
DATA_PATH = f'{PROJECT_PATH}/data/processed'
MODEL_PATH = f'{PROJECT_PATH}/outputs/models'

In [ ]:
import torch
import json
from models.dcn_v2 import (
    DCNv2, compute_quantiles_for_binning,
    create_dcn_dataloaders, load_cardinalities, load_dcn_data,
    train_dcn_model, evaluate_full
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Run preprocessing if needed
if not os.path.exists(f'{DATA_PATH}/dcn_X_train_num.parquet'):
    from preprocessing.route_dcn_prep import prepare_dcn_data
    prepare_dcn_data()

In [ ]:
# Load data
train_loader, val_loader, test_loader, _ = create_dcn_dataloaders(DATA_PATH, batch_size=1024)
cardinalities = load_cardinalities(DATA_PATH)
X_num, _, y = load_dcn_data(DATA_PATH, 'train')
num_features = list(X_num.columns)

print(f'Numerical: {len(num_features)}, Categorical: {len(cardinalities)}')

In [ ]:
# Create model (with increased dropout for regularization)
model = DCNv2(
    numerical_features=num_features,
    categorical_cardinalities=cardinalities,
    num_bins=10, numerical_embed_dim=16,
    cross_layers=3, cross_rank=64,
    deep_hidden=[512, 256, 128], 
    dropout=0.3  # Increased from 0.1 to prevent overfitting
).to(device)

# Init quantiles
for feat in num_features:
    q = compute_quantiles_for_binning(torch.tensor(X_num[feat].values), 10)
    model.embedding.numerical_binning[feat].cutpoints.data = q.to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train with Focal Loss (better for imbalanced data + hard examples)
# Early stopping based on Precision@20% (diamond recovery metric)

result = train_dcn_model(
    model, train_loader, val_loader, device,
    n_epochs=100, patience=10,
    loss_type='focal',      # Focal Loss instead of BCE
    focal_alpha=0.25,       # Weight for positive class
    focal_gamma=2.0,        # Focus on hard examples
    lr=1e-3,
    weight_decay=1e-3,      # L2 regularization
    checkpoint_path=f'{MODEL_PATH}/dcn_v2_cat_best.pth'
)

In [ ]:
# Load best and evaluate on test set
# Note: weights_only=False needed for PyTorch 2.6+ compatibility
ckpt = torch.load(f'{MODEL_PATH}/dcn_v2_cat_best.pth', weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])

test_metrics = evaluate_full(model, test_loader, device)
print(f'Test AUC: {test_metrics["auc"]:.4f}')
print(f'Test Precision@20%: {test_metrics["precision_at_k"]:.4f}')
print(f'Test Diamonds Found: {test_metrics["n_diamonds"]}')
print(f'\nRoute B baseline: AUC=0.7391, P@20%=0.965')

In [ ]:
# Save results
results = {
    'model': 'DCN-v2-categorical-focal',
    'best_epoch': result['best_epoch'],
    'val_auc': result['best_val_auc'],
    'val_precision_20': result['best_val_prec'],
    'test_auc': test_metrics['auc'],
    'test_precision_20': test_metrics['precision_at_k'],
    'test_diamonds': test_metrics['n_diamonds'],
}
with open(f'{PROJECT_PATH}/outputs/results/dcn_v2_cat_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print('Results saved.')